In [ ]:
import json
import sys
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from plotting_utils import load_in_combined_results, load_monitoring_data
from simulate_markov_utils import plot_markov_df
from sklearn.metrics import auc, roc_curve
from turn_by_turn_monitoring import plot_markov_analysis, plot_roc_curves, plot_suspiciousness_distributions

from utils.path_utils import resolve_path

sys.path.append('..')
from utils.analysis_utils import analyze_all_tasks, extract_verdict

# Constants
MAIN_AGENT_MODELS = ['gpt-4o', 'claude-3-5-sonnet-20241022']
MONITOR_AGENT_MODELS = ['gpt-4o', 'claude-3-5-sonnet-20241022', 'o3-mini', 'o1', 'claude-3-5-haiku-20241022']

# Load combined results using the cached function
include_failed_attempts = False
include_failed_main_task = True  # Assuming this is the default behavior
combined_results = load_in_combined_results(include_failed_attempts, include_failed_main_task)

# Get unique values for reference
TASK_NAMES = list(combined_results.task_name.unique())
TRANSCRIPT_TYPES = list(combined_results.transcript_type.unique())


In [ ]:

def load_turn_by_turn_data(base_dir: str = "../monitor_responses/turn_by_turn_monitoring", transcript_types: Optional[list[str]] = None):
    """Load all turn-by-turn monitoring data from the specified directory structure.
    
    Directory structure:
    base_dir/
        task_name/
            transcript_type/
                agent_model/
                    monitor_model/
    """
    # Use the load_monitoring_data function from plotting_utils.py
    return load_monitoring_data(base_dir=base_dir, transcript_types=transcript_types)

df = load_turn_by_turn_data()

In [ ]:
df['max_suspiciousness'] = df.data.apply(lambda x: max([extract_verdict(y, tag_word='verdict') for y in x.values()])) #type: ignore

In [ ]:
# Merge the dataframes
merged_df = pd.merge(
    df,
    combined_results,
    on=['task_name', 'transcript_type', 'main_agent', 'monitor_agent', 'file_name'],
    how='left'
).dropna()
merged_df['max_suspiciousness'] = np.maximum(merged_df.max_suspiciousness, merged_df.default_suspiciousness)
merged_df['convo_length'] = merged_df.transcript.apply(lambda x: len(x['conversation']))

In [ ]:
merged_df['convo_length'] = merged_df.transcript.apply(lambda x: len(x['conversation']))
merged_df['final_suspiciousness'] = merged_df.apply(lambda row: extract_verdict(row.data[str(row.convo_length)], tag_word='verdict'), axis=1) #type: ignore


In [ ]:
print("Shape of df before merge:", df.shape)
print("Shape of combined_results before merge:", combined_results.shape)
print("Shape of merged_df after merge:", merged_df.shape)

In [ ]:

# Use the plot_suspiciousness_distributions function from plotting_utils.py
plot_suspiciousness_distributions(
    merged_df, 
    monitors=['claude-3-5-haiku-20241022', 'gpt-4o'],
    metrics=['max_suspiciousness', 'final_suspiciousness']
)


In [ ]:
give_hint = False
include_failed_attempts = False
include_failed_main_task = True

# Use the plot_roc_curves function from plotting_utils.py
plot_roc_curves(
    merged_df,
    combined_results,
    give_hint=give_hint,
    include_failed_attempts=include_failed_attempts,
    include_failed_main_task=include_failed_main_task,
    monitors=['claude-3-5-haiku-20241022', 'gpt-4o'],
    metrics=['max_suspiciousness', 'final_suspiciousness', 'default_suspiciousness']
)


In [ ]:
monitor_model = 'gpt-4o'
give_hint = False

# Use the plot_markov_analysis function from plotting_utils.py
plot_markov_analysis(
    merged_df,
    main_agent='claude-3-5-sonnet-20241022',
    monitor_model=monitor_model,
    give_hint=give_hint,
    metrics=['default_suspiciousness', 'max_suspiciousness', 'final_suspiciousness']
)


In [ ]:
merged_df[(merged_df.monitor_agent == 'claude-3-5-haiku-20241022')].default_suspiciousness.value_counts()
